In [8]:
import sys
sys.path.append("..")            # 저장소 루트 (project 패키지)
sys.path.append("../scripts")    # eval_silver 변환 함수 재사용
from pathlib import Path

In [9]:
BENCH_ID = "AIHub_WelfareCounsel_counsel_clean"
SILVER   = "/data/ASR/BENCHMARK/SILVER/AIHub_WelfareCounsel/transcript.jsonl"
MODEL    = "openai/whisper-small"
DEVICE   = "cuda:2"              # 실행 직전 nvidia-smi 로 빈 GPU 확인
SAMPLE   = 1000                  # 무작위 샘플 크기 (전체 22만 중)
SEED     = 42                    # 재현용

OUT_DIR  = Path(f"../BENCHMARK/results/whisper_small__{BENCH_ID}")

In [10]:
import random
from eval_silver import convert_silver

# 1) 전체 변환 (원본 → GOLD 필드명). 원본 SILVER 는 읽기만 함.
conv_full = OUT_DIR / "_silver_converted" / f"{BENCH_ID}.jsonl"
n = convert_silver(Path(SILVER), conv_full, corpus_id=BENCH_ID)
print(f"전체 변환: {n} samples")

# 2) 무작위 SAMPLE개 추출 (seed 고정 → 재현 가능)
lines = conv_full.read_text(encoding="utf-8").splitlines()
random.seed(SEED)
sample_lines = random.sample(lines, SAMPLE)
conv = conv_full.with_name(f"{BENCH_ID}__sample{SAMPLE}.jsonl")
conv.write_text("\n".join(sample_lines) + "\n", encoding="utf-8")
print(f"무작위 샘플: {len(sample_lines)} samples → {conv}")

전체 변환: 223546 samples
무작위 샘플: 1000 samples → ../BENCHMARK/results/whisper_small__AIHub_WelfareCounsel_counsel_clean/_silver_converted/AIHub_WelfareCounsel_counsel_clean__sample1000.jsonl


In [11]:
from project.data.adapters.whisper import build_predict_fn

predict_fn = build_predict_fn(
    MODEL, backbone=MODEL,
    language="ko", task="transcribe",
    beam_size=5, batch_size=16, device=DEVICE,
)

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

In [12]:
from project.evaluation import evaluate_on_benchmark_suite

results = evaluate_on_benchmark_suite(
    model_name=f"whisper_small__{BENCH_ID}",
    predict_fn=predict_fn,
    benchmark_paths={BENCH_ID: conv},
    out_dir=OUT_DIR,
    batch_size=16,
)
results[BENCH_ID]

[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take p

CerResult(cer=11.32700267835403, scer=11.016897456231318, wer=40.392843251653055, samples=1000, per_sample_cer=[3.3333333333333335, 15.384615384615385, 10.526315789473683, 3.7037037037037033, 4.3478260869565215, 9.67741935483871, 8.333333333333332, 17.857142857142858, 10.526315789473683, 16.666666666666664, 12.5, 44.44444444444444, 6.25, 8.333333333333332, 72.22222222222221, 0.0, 7.142857142857142, 4.3478260869565215, 11.11111111111111, 8.695652173913043, 3.7037037037037033, 5.263157894736842, 13.636363636363635, 5.0, 10.0, 6.25, 7.6923076923076925, 5.555555555555555, 11.76470588235294, 6.25, 5.263157894736842, 6.25, 0.0, 11.11111111111111, 12.5, 2.5, 11.76470588235294, 33.33333333333333, 3.225806451612903, 15.0, 60.0, 26.666666666666668, 12.5, 41.17647058823529, 12.5, 12.903225806451612, 4.761904761904762, 4.3478260869565215, 11.538461538461538, 7.142857142857142, 0.0, 5.88235294117647, 0.0, 17.857142857142858, 15.384615384615385, 10.0, 13.333333333333334, 9.67741935483871, 10.0, 17.6

In [13]:
print((OUT_DIR / "evaluation_report.txt").read_text(encoding="utf-8"))

📊 ASR Evaluation Report — whisper_small__AIHub_WelfareCounsel_counsel_clean
   Date: 2026-06-16T17:10:36

## 1. Benchmark Set Results (한국어 CER 표준)
--------------------------------------------------------------------------------
Benchmark                                                  CER (%)   sCER (%)    Samples
--------------------------------------------------------------------------------
AIHub_WelfareCounsel_counsel_clean                           11.33      11.02      1,000
--------------------------------------------------------------------------------
Weighted Average                                             11.33                 1,000

## 2. Slice Analysis (메타 필드별)
--------------------------------------------------------------------------------

### AIHub_WelfareCounsel_counsel_clean
  [by age_group]
  value                   CER (%)    samples
  60대                       12.86        142
  40대                       12.23        297
  30대                       10.61      

In [14]:
import json
import pandas as pd
import jiwer

lines = (OUT_DIR / BENCH_ID / "predictions.jsonl").read_text(encoding="utf-8").splitlines()
df = pd.DataFrame(json.loads(l) for l in lines if l)

df["cer"] = [
    jiwer.cer(r, h) * 100 if r else float("nan")
    for r, h in zip(df["text_normalized"], df["prediction_normalized"])
]

for _, row in df.sort_values("cer", ascending=False).head(20).iterrows():
    print(f"[CER {row.cer:5.1f}] 정답: {row.text_normalized}")
    print(f"             예측: {row.prediction_normalized}\n")

[CER 1100.0] 정답: Z
             예측: 그거 너무 다행이예요

[CER  72.2] 정답: 000 0000 0000 입니다.
             예측: 공공공 공공공 공공공입니다

[CER  66.7] 정답: ㅇㅇㅇ 이요
             예측: 공공공이요

[CER  60.0] 정답: 어떻게요?
             예측: 어떻게 해요

[CER  60.0] 정답: 어떻게요?
             예측: 어떻게 해요

[CER  57.1] 정답: 수고하십니다.
             예측: 도와주십니다

[CER  57.1] 정답: ㅇㅇㅇ이에요.
             예측: 공공공이에요

[CER  56.2] 정답: 시간 당 0,000 원입니다.
             예측: 뛰간당 0-0-0-0-5 입니다

[CER  55.6] 정답: 그러려니 했어요.
             예측: 브러리언이 했어요

[CER  55.6] 정답: ㅇㅇ역 근처에요.
             예측: 몽공력 근처예요

[CER  54.5] 정답: 상담사 ㅇㅇㅇ입니다.
             예측: 땅담사 0000입니다

[CER  53.3] 정답: ㅇㅇ는 착하고 사려 깊어요.
             예측: 공공은 착하고 사력이 퍼요

[CER  52.9] 정답: 네, ㅇㅇㅇ동 ㅇㅇ백화점이군요.
             예측: 네 000000 백화점이군요

[CER  50.0] 정답: 네네
             예측: 네 네

[CER  47.1] 정답: ㅇㅇㅇ동 ㅇㅇㅇ 빌라로 갈거에요
             예측: 공공공동 공공공 빌라로 갈 거예요

[CER  46.7] 정답: 네, ㅇㅇㅇ상담원이었습니다.
             예측: 네 0000 상담원이었습니다

[CER  46.2] 정답: 예약실 ㅇㅇㅇ이였습니다.
             예측: 예약실 0000이었습니다

[CER  46.2] 정답: 한 두 해 정도 후에요.
       